In [ ]:
# Hier nichts ändern, nur ausführen!
import sys; sys.path.append("../code"); from setup_AB6 import *; setup_erfolg();

<div>
    <img src="../figs/web-design.png" style="height:8em; width:auto; float: right; margin-right: 10px;">
</div>

# Arbeitsblatt 6 | Projekt: Lights-out mit Grover

Willkommen in der letzten Woche von HiSim! Heute starten wir unser Abschlussprojekt: Wir werden das klassische Spiel **Lights-out** mit dem Grover-Algorithmus lösen!

In dieser Woche werdet ihr in Dreier- oder Vierergruppen arbeiten.

**Lernziele:**
- Ihr wendet den Grover-Algorithmus auf ein praktisches Problem an
- Ihr versteht, wie man ein Problem als Suchproblem formuliert
- Ihr implementiert ein vollständiges Projekt

**Zeitaufwand:** ca. 3 Stunden (Aufgaben in Blöcken von max. 30 Minuten)

**Voraussetzung:** Arbeitsblätter [1](./AB1_Qubits.ipynb), [2](./AB2_Gates.ipynb), [3](./AB3_Entanglement.ipynb), [4](./AB4_DeutschJozsa.ipynb) und [5](./AB5_Grover.ipynb)

## 1) Das Spiel Lights-out

**Lights-out** ist ein Puzzle-Spiel:
- Ihr habt ein Gitter von Lichtern (z.B. 3x3)
- Wenn ihr auf ein Licht klickt, werden dieses Licht und seine direkten Nachbarn (links, rechts, oben, unten) umgeschaltet
- **Ziel:** Alle Lichter ausschalten (alle auf 0 setzen)

**Beispiel:**

<img src="../figs/lights_out_concept.png" width="70%"/>

Die gelb eingefärbten Flächen sind diejenigen, bei denen das Licht eingeschalten ist. Bei den grauen, ist das Licht bereits aus. Im Beispiel wird nun auf die mittlere Fläche geklickt. Bei diesem Licht und allen direkten Nachbarn (dunkelblau umrandet), wird nun der Status des Lichts geändert. Wenn das Licht vorher an war, ist es hinterher aus und andersrum.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Spielt Lights-Out und notiert eure Beobachtungen  </h3>
  
  Im Folgenden findet ihr ein kleines Lights-Out-Spiel. Spielt ein paar Runden und schaut euch insbesondere die Anzahl eurer Klicks und eurer Klick-Historie an. Welche Beobachtungen lassen sich daraus ableiten?
    
</div>

In [ ]:
# Starte das Spiel!
game = LightsOutGame(size=3)

In [ ]:
show_multi_selection_1a()

## 2) Das Problem als Suchproblem

Bei einem 3x3-Gitter gibt es $2^9 = 512$ mögliche Lösungen.

**Klassisch** müsstet ihr alle Möglichkeiten durchprobieren. Das schauen wir uns zunächst an, bevor wir uns eine Lösung mit Hilfe des Grover Algorithmus überlegen.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Lights-out simulieren  </h3>

  Wir wollen die klassische Variante von Lights-out simulieren. Dazu erstellen wir uns zunächst ein paar notwendige Funktionen. Eure Aufgabe ist es, eine Funktion zu implementieren, die die korrekten Nachbarn eines vorgegebenes Feld und dem Feld selbst zurück gibt.
  Erstellt eine Funktion, die das Spiel simuliert:
  <ol>
    <li>Erstellt eine Funktion, die alle validen Nachbarn und das Feld selbst zurück gibt</li>
    <li>Testet mit einem einfachen Beispiel</li>
  </ol>

  Bitte beachtet, dass ein Feld am Rand des Spielfelds weniger Nachbarn hat.

  <hr>
    <a href="../help/tipps_AB6_2a.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Tipp</a> <span style="margin-right: 20px;"></span>
    <a href="../help/zusatz_AB6_2a.ipynb"><i class="fas fa-puzzle-piece" style="font-size:20px"></i> &nbsp;Zusatzaufgabe</a>
</div>

In [ ]:
import numpy as np

# Spielfeld: 3x3 Gitter
# 1 = Licht an, 0 = Licht aus

def zeige_feld(feld):
    """Zeigt das Spielfeld"""
    for zeile in feld:
        print(' '.join(['■' if x else '□' for x in zeile]))

def nachbarn(x, y, size=3):
    """Bestimmt die validen Nachbarn von Feld x, y (inklusive x, y)"""
    nachbarn = [[x, y]]

    # Euer Code: Bestimmt die vier Nachbarn (oben, unten, links, rechts)

    return nachbarn

def zug(feld, x, y):
    """Führt einen Zug aus: Togglet das Feld und seine Nachbarn"""
    kopie = [row[:] for row in feld]  # Kopie erstellen

    for nx, ny in nachbarn(x, y):
        kopie[nx][ny] = 1 - kopie[nx][ny]
    
    return kopie

def alle_aus(feld):
    """Prüft, ob alle Lichter aus sind"""
    for zeile in feld:
        if not all(licht == 0 for licht in zeile):
            return False
    return True

# Teste
feld = [[1, 1, 0],
        [1, 1, 1],
        [0, 1, 0]]

print("Startfeld:")
zeige_feld(feld)

# Führt einen Zug aus
neues_feld = zug(feld, 1, 1)
print("\nNach Zug auf (1,1):")
zeige_feld(neues_feld)

print(f"\nAlle aus? {alle_aus(neues_feld)}")

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> b) Klassische Lösung  </h3>
  Löst das Problem klassisch:
  <ol>
    <li>Probiert alle möglichen Kombinationen durch</li>
    <li>Findet eine Lösung für das Beispiel</li>
    <li>Wie viele Versuche braucht ihr?</li>
  </ol>
  <hr>
    <a href="../help/tipps_AB6_2b.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Tipp</a> <span style="margin-right: 20px;"></span>
    <a href="../help/zusatz_AB6_2b.ipynb"><i class="fas fa-puzzle-piece" style="font-size:20px"></i> &nbsp;Zusatzaufgabe</a>
</details>
</div>

In [ ]:
from itertools import product

def combo_zu_koordinaten(combo):
    """Hilfsfunktion, die aus der Combo eine Liste der zu klickenden x, y-Koordinate macht"""
    koordinaten = []
    laenge = int(np.sqrt(len(combo)))
    for index, value in enumerate(combo):
        if value == 1:
            koordinaten.append(divmod(index, laenge))
    return koordinaten

def klassische_loesung(feld):
    """Löst das Problem klassisch"""
    # Alle möglichen Kombinationen von Zügen (9 mögliche Klicks bei 3x3 Feld)
    # Euer Code: Probiert alle 2**9 == 512 Kombinationen durch
    
    for combo in product([0, 1], repeat=9):
        kopie = [row[:] for row in feld]  # Kopie erstellen
        # Euer Code: Wendet die Kombination auf das Feld an
        # Euer Code: Prüft, ob alle Lichter aus sind
    
    return None  # Keine Lösung gefunden

# Teste
feld = [[0, 1, 0],
        [1, 1, 1],
        [0, 1, 0]]

print("Suche nach klassischer Lösung...")
import time
start = time.time()
loesung = klassische_loesung(feld)
ende = time.time()

print(f"Zeit: {ende - start:.4f} Sekunden")
if loesung:
    print(f"Lösung gefunden: {loesung}")

## 3) Grover für Lights-out

Jetzt werden wir Grover nutzen, um das Problem zu lösen!
Mit Hilfe des **Grover** Algorithmus können wir die richtige Kombination von Klicks *suchen*. Dazu brauchen wir ca. $\sqrt{512} \approx 23$ Versuche anstelle 512!

**Idee:**
1. Kodiere die Kombination aller möglichen Klicks in Qubits (9 Qubits = 512 Möglichkeiten)
2. Erstelle ein Orakel, das prüft, ob alle Lichter aus sind
3. Nutze Grover, um die Lösung zu finden

<div id="information" class="alert alert-success">
  <h2><i class="fas fa-info" style="font-size:36px"></i> &nbsp;  Lights-out mit Grover</h2>
    <ol>
        <li><span> <strong>Kodierung:</strong> 9 Qubits für 9 mögliche Klicks und 9 Qubits für das Feld</span></li>
        <li><span><strong>Orakel:</strong> Prüft, ob alle Lichter nach den Klicks aus sind</span></li>
        <li><span><strong>Grover:</strong> Verstärkt die Amplitude der Lösung</span></li>
        <li><span><strong>Messung:</strong> Lest die Lösung ab</span></li>
    </ol>
</div>

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Lights-out Orakel  </h3>
  Erstellt das Orakel für Lights-out:
  <ol>
    <li>Erstellt ein Orakel, das prüft, ob alle Lichter aus sind</li>
    <li>Das Orakel soll die Phase flippen, wenn die Lösung gefunden wurde</li>
  </ol>

  Das Orakel ist für Lights-Out etwas komplexer, als wir es beim Grover letzte Woche kennengelernt haben. Wir müssen einerseits einen Bereich, das `input_register`, im Circuit definieren, indem wir unser Feld kodieren. Zusätzlich brauchen wir einen Suchbereich, `such_register`, in dem wir unsere Superposition erzeugen, um alle möglichen Klicks abbilden zu können.

  Diese beiden Bereiche werden durch die Klicks, die ihr darauf definiert, stark miteinander verschränkt. Daher werden uns diesmal einen zusätzlichen Bereich für den Phasenflip anlegen, das `phase_register`. Hierfür wird aber nur ein Qubit benötigt.

  Eure Aufgabe ist es also zu überlegen, wie genau ihr die einzelnen Klicks abbildet. Wir würden euch dafür die Nutzung des `CX`-Gates empfehlen. Der eigentliche Test für das Orakel testet dann auf den 0-Zustand, also ob alle Lichter aus sind. Dies habt ihr analog bereits in der letzten Woche gemacht.

  <hr>
    <a href="../help/tipps_AB6_3a.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Tipp</a> <span style="margin-right: 20px;"></span>
</div>

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister

start_feld = [[0, 1, 0],
              [1, 1, 1],
              [0, 1, 0]]

input_register = QuantumRegister(len(start_feld)**2, "data")
such_register = QuantumRegister(len(start_feld)**2, "oracle")
phase_register = QuantumRegister(1, "phase")

orakel_circuit = QuantumCircuit(input_register, such_register, phase_register)

def qubit_zu_koordinate(qubit):
    """Hilfsfunktion, die den Qubit-Index zur Koordinate umwandelt"""
    return divmod(qubit, len(start_feld))

def koordinate_zu_qubit(x, y):
    """Hilfsfunktion, die eine Koordinate zu einem Qubit-Index umwandelt"""
    return x * len(start_feld) + y

def init_input(feld, circuit, register):
    """Funktion, die das Feld als Input auf einem Register initialisiert"""
    # Euer Code: Wenn im Feld ein Wert 1 ist, dann setzt ein X-Gate


def lights_out_oracle(circuit, data, suche, phase):
    """Erstellt ein Orakel für Lights-out"""
    
    # Euer Code: Simuliert alle möglichen Klicks
    # Für jede mögliche Kombination:
    #   - Prüfe, ob alle Lichter aus sind
    #   - Wenn ja, flippe die Phase
    # Macht die Klicks wieder rückgängig
    pass

# Teste das Orakel
init_input(start_feld, orakel_circuit, input_register)
suche = lights_out_oracle(orakel_circuit, input_register, such_register, phase_register)

print(orakel_circuit.draw())

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> b) Grover für Lights-out  </h3>
  Implementiere den vollständigen Grover-Algorithmus:
  
  1. Initialisiere alle Qubits im `such_register` in Superposition
  2. Initialisiere das `phase_register` mit Hadamard
  3. Wende das Orakel auf das `such_register` und `phase_register` an
  4. Wende die Diffusion auf das `such_register` und `phase_register` an
  5. Messe das Ergebnis und teste mit unterschiedlichen Anzahlen von Iterationen

  <hr>
    <a href="../help/tipps_AB6_3b.ipynb"><i class="fas fa-life-ring" style="font-size:20px"></i> &nbsp;Tipp</a> <span style="margin-right: 20px;"></span>
    <a href="../help/zusatz_AB6_3b.ipynb"><i class="fas fa-puzzle-piece" style="font-size:20px"></i> &nbsp;Zusatzaufgabe</a>
</div>

In [ ]:
from qiskit_aer import AerSimulator
import math
from qiskit import ClassicalRegister

start_feld = [[0, 1, 0],
              [1, 1, 1],
              [0, 1, 0]]

input_register = QuantumRegister(len(start_feld)**2, "data")
such_register = QuantumRegister(len(start_feld)**2, "oracle")
phase_register = QuantumRegister(1, "phase")
classical_register = ClassicalRegister(len(start_feld)**2, "output")

grover_circuit = QuantumCircuit(input_register, such_register, phase_register, classical_register)

def diffusion(circuit, suche, phase):
    """Diffusion zur Amplituden-Verstärkung"""
    # Euer Code: Implementiert die Diffusion
    pass

def grover_lights_out(circuit, data, suche, phase, iterationen=1):
    """Löst Lights-out mit Grover"""
    
    # Euer Code: Initialisiert das Such- und Phase-Register
    
    # Euer Code: Grover-Iterationen
    for _ in range(iterationen):
        # Orakel
        # Diffusion
        pass
    

# Euer Code: Initialisert das Input-Register

grover_lights_out(grover_circuit, input_register, such_register, phase_register, 1)

# Euer Code: Messt das korrekte Register
# grover_circuit.measure(None, classical_register)

grover_circuit.reverse_bits()  # Umdrehen der Codierung zum korrekten Ablesen

print("Grover für Lights-out:")
print(grover_circuit.draw())

# Simulation
simulator = AerSimulator()
job = simulator.run(grover_circuit, shots=100)
result = job.result()
counts = result.get_counts()
print(counts)

print("\nTop 3 Ergebnisse:")
for state, count in sorted(counts.items(), key=lambda x: -x[1])[:3]:
    print(f"  {state}: {count}%")

## 4) Ergebnisse auswerten

Die Messergebnisse zeigen, welche Qubits (Felder) du klicken sollst:

- Qubit 0 = Feld (0,0)
- Qubit 1 = Feld (0,1)
- ...
- Qubit 8 = Feld (2,2)

**Wenn ein Qubit 1 ist**, sollst du auf dieses Feld klicken!

## 4) Projekt abschließen

Jetzt seid ihr dran! Vervollständigt das Projekt:

**Aufgaben:**
1. Testet verschiedene Startkonfigurationen
2. Vergleicht die Ergebnisse mit der klassischen Lösung
3. Erstellt eine Visualisierung der Lösung
4. Überlegt, wie sich Noise auf die Ergebnisse auswirken könnte

**Fragen zum Nachdenken:**
- Wie unterscheiden sich die Ergebnisse bei verschiedenen Konfigurationen?
- Ist die klassische oder die Quantenlösung schneller?
- Wie stark beeinflusst Noise die Ergebnisse?

In [ ]:
# Euer Projekt:

# 1. Testet verschiedene Konfigurationen
konfigurationen = [
    [[1, 1, 0], [1, 1, 1], [0, 1, 0]],  # Beispiel 1
    [[0, 1, 0], [1, 1, 1], [0, 1, 0]],  # Beispiel 2
    [[1, 0, 1], [0, 1, 0], [1, 0, 1]],  # Beispiel 3
    # Dein Code: Fügt weitere Konfigurationen hinzu
]

for i, feld in enumerate(konfigurationen):
    print(f"\n=== Konfiguration {i+1} ===")
    zeige_feld(feld)
    
    # Dein Code: Löse mit Grover
    # qc = grover_lights_out(feld, iterationen=...)
    # simulator = AerSimulator()
    # job = simulator.run(qc, shots=100)
    # counts = job.result().get_counts()
    # print(counts)

# 2. Vergleiche mit klassischer Lösung
# Dein Code hier:

# 3. Visualisierung
# Dein Code hier:

## Zusammenfassung Woche 6

In dieser Woche haben wir gelernt:

1. **Lights-out:** Ein Puzzle-Spiel, bei dem man alle Lichter ausschalten muss.

2. **Problemformulierung:** Das Problem kann als Suchproblem formuliert werden.

3. **Grover-Anwendung:** Der Grover-Algorithmus findet die Lösung in $\sqrt{N}$ statt $N$ Versuchen.

4. **Orakel-Erstellung:** Das Orakel prüft, ob alle Lichter nach den Klicks aus sind.

5. **Projekt:** Ihr habt ein vollständiges Projekt mit Quantencomputing gelöst!

<hr>

## Abschluss <i class="fa fa-trophy" style="font-size:36px"></i> 

Herzlichen Glückwunsch! Ihr habt nun alle Arbeitsblätter abgeschlossen!

Ihr habt so einiges gelernt:
- Qubits und Superposition
- Quanten-Gatter und Schaltkreise
- Verschränkung und Bell States
- Deutsch-Jozsa Algorithmus
- Grover-Suche
- Projekt: Lights-out mit Grover

**Nächste Schritte:**
- Experimentiert weiter mit Qiskit!
- Schaut euch weitere Quantenalgorithmen an!
- Informiert euch über echte Quantencomputer!

___
<hr>
<img src = "../figs/CC-BY-SA.png" style="height:3em; width:auto"> 
Dieses Werk ist lizenziert unter einer <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/">Creative Commons Namensnennung - Weitergabe unter gleichen Bedingungen 4.0 International Lizenz</a>. </br>
Autoren: Eileen Kühn, Gabriel Mejia Ruiz, SCC/KIT
&nbsp;

<sup>1</sup> Qiskit Documentation: <a href="https://qiskit.org/documentation/">https://qiskit.org/documentation/</a>